# 🛡️ Anti-Fraud Detection - Google Colab

## Installation des dépendances

In [ ]:
# Installer PySpark
!pip install pyspark==3.4.1

# Installer autres dépendances
!pip install pandas matplotlib seaborn scikit-learn

## Importer les bibliothèques

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, max as spark_max, min as spark_min
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("✅ Bibliothèques importées avec succès!")

## Créer la session Spark

In [ ]:
# Créer session Spark optimisée pour Colab
spark = SparkSession.builder \
    .appName("AntiFraud-Colab") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "12g") \
    .getOrCreate()

print(f"✅ Session Spark créée!")
print(f"Version Spark: {spark.version}")

##Uploader le dataset PaySim

**Instructions:**
1. Téléchargez le dataset depuis: https://www.kaggle.com/ealaxi/paysim1
2. Cliquez sur l'icône 📁 dans la barre latérale gauche
3. Cliquez sur "Importer"
4. Sélectionnez votre fichier CSV
5. Le fichier apparaîtra dans `/content/`

In [ ]:
# Vérifier si le dataset est uploadé
import os

# Lister les fichiers dans /content/
print("📁 Fichiers dans Colab:")
for file in os.listdir('/content/'):
    if file.endswith('.csv'):
        print(f"   📊 {file}")
        dataset_path = f'/content/{file}'

if 'dataset_path' in locals():
    print(f"\n✅ Dataset trouvé: {dataset_path}")
else:
    print("\n❌ Dataset non trouvé. Veuillez uploader le fichier CSV!")

## Charger et analyser le dataset

In [ ]:
# Charger le dataset
if 'dataset_path' in locals():
    print(f"🚀 Chargement du dataset...")
    start_time = time.time()
    
    df = spark.read.csv(dataset_path, header=True, inferSchema=True)
    
    load_time = time.time() - start_time
    print(f"✅ Dataset chargé en {load_time:.2f} secondes")
    print(f"📊 Dimensions: {df.count():,} lignes × {len(df.columns)} colonnes")
    
    # Afficher le schéma
    print("\n📋 Schéma du dataset:")
    df.printSchema()
else:
    print("❌ Veuillez d'abord uploader le dataset!")

## Analyse des fraudes

In [ ]:
if 'df' in locals():
    # Analyse du taux de fraude
    print("🔍 Analyse des fraudes:")
    
    fraud_stats = df.groupBy("isFraud").agg(
        count("*").alias("transaction_count"),
        (count("*") / df.count() * 100).alias("percentage")
    ).orderBy("isFraud")
    
    fraud_stats.show()
    
    # Analyse par type de transaction
    print("\n📈 Analyse par type:")
    type_stats = df.groupBy("type").agg(
        count("*").alias("total_transactions"),
        spark_sum("isFraud").alias("fraud_count"),
        avg("amount").alias("avg_amount")
    ).orderBy("total_transactions", ascending=False)
    
    type_stats.show(truncate=False)
    
    # Montants par statut fraude
    print("\n💰 Montants par statut:")
    amount_stats = df.groupBy("isFraud").agg(
        avg("amount").alias("avg_amount"),
        spark_max("amount").alias("max_amount"),
        spark_min("amount").alias("min_amount")
    ).orderBy("isFraud")
    
    amount_stats.show(truncate=False)

## Tests de performance Colab

In [ ]:
if 'df' in locals():
    print("⚡ Tests de performance Colab:")
    
    # Test 1: Agrégation simple
    start_time = time.time()
    simple_agg = df.groupBy("type").count().collect()
    simple_time = time.time() - start_time
    print(f"   Agrégation simple: {simple_time:.2f}s")
    
    # Test 2: Agrégation complexe
    start_time = time.time()
    complex_agg = df.groupBy("type", "isFraud").agg(
        avg("amount"),
        count("*")
    ).collect()
    complex_time = time.time() - start_time
    print(f"   Agrégation complexe: {complex_time:.2f}s")
    
    # Test 3: Filtrage
    start_time = time.time()
    fraud_filter = df.filter(col("isFraud") == 1).count()
    filter_time = time.time() - start_time
    print(f"   Filtrage fraude: {filter_time:.2f}s")
    
    print(f"\n🎯 Fraudes détectées: {fraud_filter:,}")

## Visualisations

In [ ]:
if 'df' in locals():
    # Convertir en Pandas pour visualisation
    pandas_df = df.sample(fraction=0.1, seed=42).toPandas()
    
    # Graphique des types de transactions
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    pandas_df['type'].value_counts().plot(kind='bar')
    plt.title('Types de transactions')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 2, 2)
    fraud_by_type = pandas_df.groupby('type')['isFraud'].sum()
    fraud_by_type.plot(kind='bar', color='red')
    plt.title('Fraudes par type')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Visualisations générées!")

## Nettoyage

In [ ]:
# Arrêter Spark
if 'spark' in locals():
    spark.stop()
    print("✅ Session Spark arrêtée")

print("\n🎉 Analyse terminée dans Google Colab!")
print("\n💡 Prochaines étapes:")
print("   1. Feature engineering")
print("   2. Modèles de Machine Learning")
print("   3. API de prédiction")